# Notebook 04: ArcGIS Web Maps and Dashboard

**Publishing the Lake Mead clustering results and building an interactive comparison dashboard**

Notebook 03 produced three CSVs in `data/processed/`:

- `mead_fires_clustered.csv`: 844 wildfires with cluster labels from each method
- `mead_cluster_summary.csv`: per-cluster summary stats for both methods
- `mead_global_metrics.csv`: headline metrics in a single row

This notebook uploads them to ArcGIS Online, publishes the fires CSV as a hosted feature layer, and walks through building two web maps (one styled by standard $k$-means clusters, one by obstacle-aware clusters) and a dashboard that puts them side by side.

The dashboard's story: same fires, two clusterings. The left map shows what standard $k$-means produces; the right shows what obstacle-aware $k$-means produces with $\beta = 1.85$ on Lake Mead's basin. The visible difference between the two panels is the case for obstacle-aware methods on a long, narrow waterbody.

## 1. Setup

In [13]:
# Standard packages
import getpass
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon

# ArcGIS Python API -- only what we need for content management
from arcgis.gis import GIS

# Cluster colors from the project palette (reference for the Map Viewer step)
CLUSTER_COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
print('Cluster colors for the Map Viewer:')
for i, c in enumerate(CLUSTER_COLORS, start=1):
    print(f'  Cluster {i}: {c}')

Cluster colors for the Map Viewer:
  Cluster 1: #e74c3c
  Cluster 2: #3498db
  Cluster 3: #2ecc71
  Cluster 4: #f39c12


## 2. Loading Results from Notebook 03

Three CSVs feed the dashboard. The fires CSV becomes a hosted feature layer with points on the map; the cluster summary and global metrics become hosted tables that drive the dashboard's stat indicators.

In [2]:
processed_dir = Path('../data/processed')

fires = pd.read_csv(processed_dir / 'mead_fires_clustered.csv')
cluster_summary = pd.read_csv(processed_dir / 'mead_cluster_summary.csv')
global_metrics = pd.read_csv(processed_dir / 'mead_global_metrics.csv')

print(f'Loaded {len(fires)} fires')
print(f'Cluster summary rows: {len(cluster_summary)} (4 std + 4 oa_opt)')
print(f'Global metrics rows: {len(global_metrics)}')
print()
print('Cluster columns available in fires:')
for col in fires.columns:
    if 'cluster' in col:
        print(f'  {col}')

Loaded 844 fires
Cluster summary rows: 8 (4 std + 4 oa_opt)
Global metrics rows: 1

Cluster columns available in fires:
  cluster_basin_opt
  cluster_basin_std
  cluster_basin_eq
  cluster_near_std
  cluster_near_eq
  cluster_near_opt


In [3]:
# Headline numbers for the dashboard indicators
global_metrics.round(4)

,lake,total_fires,year_min,year_max,k,opt_beta_basin,span_std_basin,span_oa_opt_basin,span_improvement_pct_basin
0,Mead,844,1992,2020,4,1.85,0.3371,0.3046,9.6346


In [4]:
# Per-cluster numbers for the dashboard side panels
cluster_summary.round(2)

,method,cluster_id,cluster_label,n_fires,mean_fire_size,median_fire_size,pct_human,pct_natural,cluster_span
0,std,0,1,185,335.52,0.40,16.76,83.24,0.38
1,std,1,2,124,69.32,0.20,72.58,27.42,0.35
2,std,2,3,273,1234.41,1.50,6.23,93.77,0.29
3,std,3,4,262,4.96,0.10,81.30,18.70,0.33
4,oa_opt,0,1,181,318.25,0.40,16.02,83.98,0.35
5,oa_opt,1,2,126,69.30,0.20,73.02,26.98,0.27
6,oa_opt,2,3,278,1228.73,1.75,6.83,93.17,0.29
7,oa_opt,3,4,259,4.01,0.10,81.47,18.53,0.31


## 3. Preparing Data for ArcGIS

The CSV that gets uploaded needs:

- Latitude and longitude (these drive the geocoding)
- 1-indexed cluster columns for the two methods we'll show on the maps
- Friendly cause and fire size labels
- Year as a string so ArcGIS doesn't render it with comma separators

The raw CSV from Notebook 03 has more columns than we need for the dashboard. We trim down to a publishing-ready subset.

In [5]:
# Work on a copy so we don't mutate the loaded DataFrame
fires_pub = fires.copy()

# 1-indexed cluster labels (the dashboard displays Cluster 1, 2, 3, 4)
fires_pub['Cluster_Std'] = (fires_pub['cluster_basin_std'] + 1).astype(int)
fires_pub['Cluster_OA'] = (fires_pub['cluster_basin_opt'] + 1).astype(int)

# Cause labels
fires_pub['Cause'] = fires_pub['cause_binary'].map({0: 'Natural', 1: 'Human-Caused'})
fires_pub['Specific_Cause'] = fires_pub['NWCG_GENERAL_CAUSE'].replace('Natural', 'Lightning')

# Round fire size for popup readability
fires_pub['Fire_Size_Acres'] = fires_pub['FIRE_SIZE'].round(2)

# Cast year to string so ArcGIS doesn't render it as "2,003"
fires_pub['Fire_Year'] = fires_pub['FIRE_YEAR'].astype(int).astype(str)

# Columns to publish
publish_cols = [
    'LATITUDE', 'LONGITUDE',
    'Cluster_Std',
    'Cluster_OA',
    'Cause', 'Specific_Cause',
    'Fire_Size_Acres', 'Fire_Year',
]
fires_pub = fires_pub[publish_cols].copy()

# Save to a publishable CSV
publish_path = processed_dir / 'mead_fires_webmap.csv'
fires_pub.to_csv(publish_path, index=False)

print(f'Saved {len(fires_pub)} fires to {publish_path.name}')
print()
print('Column dtypes:')
print(fires_pub.dtypes)
print()
print('Standard k-means cluster distribution:')
print(fires_pub['Cluster_Std'].value_counts().sort_index().to_string())
print()
print('OA optimized cluster distribution:')
print(fires_pub['Cluster_OA'].value_counts().sort_index().to_string())

Saved 844 fires to mead_fires_webmap.csv

Column dtypes:
LATITUDE           float64
LONGITUDE          float64
Cluster_Std          int64
Cluster_OA           int64
Cause               object
Specific_Cause      object
Fire_Size_Acres    float64
Fire_Year           object
dtype: object

Standard k-means cluster distribution:
Cluster_Std
1    185
2    124
3    273
4    262

OA optimized cluster distribution:
Cluster_OA
1    181
2    126
3    278
4    259


## 4. Connecting to ArcGIS Online

The script uploads to your AGOL organization using the ArcGIS Python API. Authentication uses a username and password prompt; the password isn't stored anywhere.

In [6]:
# Authenticate with ArcGIS Online
gis = GIS(
    url='https://michele-75.maps.arcgis.com',
    username='Michele-75',
    password=getpass.getpass('Enter your ArcGIS Online password: '),
)

print(f'Connected as: {gis.users.me.username}')
print(f'Organization: {gis.properties.name}')

Connected as: Michele-75
Organization: Michele Perry


## 5. Publishing the Hosted Items

Three items get uploaded:

1. **Fires feature layer** -- 844 points, one per fire, with cluster columns for both methods.
2. **Cluster summary table** -- 8 rows (4 per method) with per-cluster stats for the dashboard side panels.
3. **Global metrics table** -- 1 row with headline numbers.

By default the script does NOT delete existing items with the same titles. If you've published these before, the new uploads will be created alongside (with " (1)", " (2)" appended). To delete and recreate cleanly, change `delete_existing` to `True`.

If you'd rather update existing items in place (preserves item IDs and any dashboard configurations that reference them), use the Update Data flow at the bottom of this section instead.

In [7]:
delete_existing = False

# Titles we'll use for the items we're about to create
item_titles = [
    'Lake Mead Wildfires - Obstacle-Aware vs Standard k-Means',  # CSV + feature layer
    'Lake Mead Wildfires - Cluster Summary',                     # supporting table
    'Lake Mead Wildfires - Global Metrics',                      # supporting table
]

if delete_existing:
    me = gis.users.me.username
    for title in item_titles:
        # Restrict to our own items so we don't accidentally hit something shared
        existing = gis.content.search(
            query=f'title:"{title}" owner:{me}',
            max_items=20,
        )
        # Search is fuzzy -- filter to exact title match
        existing = [it for it in existing if it.title == title]
        for it in existing:
            print(f'Deleting old item: {it.title} ({it.type}, ID {it.id})')
            it.delete()
    print('Cleanup complete.\n')
else:
    print('Skipping cleanup. New items will be created alongside any existing duplicates.\n')

Skipping cleanup. New items will be created alongside any existing duplicates.



### 5.1 Publish the Fires Feature Layer

The CSV gets uploaded, then published as a hosted feature layer. ArcGIS detects the `LATITUDE` and `LONGITUDE` columns and geocodes the points automatically.

In [8]:
# Item properties for the CSV (the resulting feature layer inherits these)
fire_item_properties = {
    'title': 'Lake Mead Wildfires - Obstacle-Aware vs Standard k-Means',
    'description': (
        'Wildfire occurrence data (1992-2020) within a bounding box around Lake Mead, '
        'clustered two ways for direct comparison: '
        '<b>standard k-Means</b> (geographic coordinates only) and '
        '<b>obstacle-aware k-Means</b> (geographic coordinates plus arc-length parameter '
        '<i>s</i> along the Lake Mead shoreline, with optimized weight β = 1.85). '
        'Each fire has two cluster labels, one per method, so a dashboard can compare '
        'the two clusterings side by side.<br><br>'
        '<b>Source:</b> FPA FOD 6th Edition (USFS Research Data Archive).<br>'
        '<b>Method:</b> See project repository for full methodology, including how the '
        'optimal β was selected from the J / span objective surface.'
    ),
    'tags': 'wildfires, clustering, Lake Mead, obstacle-aware, k-means, portfolio',
    'type': 'CSV',
}

print('Uploading fire data to ArcGIS Online...')
fire_csv_item = gis.content.add(fire_item_properties, data=str(publish_path))
print(f'CSV uploaded: {fire_csv_item.title} (ID: {fire_csv_item.id})')

# Publish as a hosted feature layer.
# ArcGIS detects LATITUDE/LONGITUDE and geocodes the points automatically.
print('\nPublishing as hosted feature layer...')
fire_layer_item = fire_csv_item.publish()
print(f'Published: {fire_layer_item.title}')
print(f'Item URL: {fire_layer_item.homepage}')

Uploading fire data to ArcGIS Online...


c:\Users\mpp24\anaconda3\envs\obstacle-clustering\Lib\site-packages\IPython\core\interactiveshell.py:3701: DeprecatedWarning: add is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.add()` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


CSV uploaded: Lake Mead Wildfires - Obstacle-Aware vs Standard k-Means (ID: 8f885c0a23324bf18c660256e0d37fa0)

Publishing as hosted feature layer...
Published: Lake Mead Wildfires - Obstacle-Aware vs Standard k-Means
Item URL: https://Michele-75.maps.arcgis.com/home/item.html?id=6c75d6ddde1548d3aba8875a0efafcc0


### 5.2 Publish the Supporting Tables

Cluster summary and global metrics get uploaded as plain CSVs and published as hosted tables (no geocoding needed since they have no spatial columns).

In [9]:
# Cluster summary table
summary_item_props = {
    'title': 'Lake Mead Wildfires - Cluster Summary',
    'description': (
        'Per-cluster summary statistics for the Lake Mead clustering, with 4 rows for '
        'standard k-Means (method = "std") and 4 rows for obstacle-aware optimized '
        '(method = "oa_opt"). Includes n_fires, mean and median fire size, percent '
        'human-caused, and per-cluster arc-length span. Used to populate the dashboard '
        'side panels.'
    ),
    'tags': 'wildfires, clustering, Lake Mead, summary, dashboard',
    'type': 'CSV',
}
summary_csv_path = processed_dir / 'mead_cluster_summary.csv'
summary_csv_item = gis.content.add(summary_item_props, data=str(summary_csv_path))
summary_table_item = summary_csv_item.publish()
print(f'Cluster summary published: {summary_table_item.title}')
print(f'  {summary_table_item.homepage}')

# Global metrics table
metrics_item_props = {
    'title': 'Lake Mead Wildfires - Global Metrics',
    'description': (
        'Headline metrics for the Lake Mead clustering. Single row with total fire count, '
        'year range, optimal β, mean arc-length span for the standard and optimized '
        'methods, and the percent span improvement. Used to populate the dashboard '
        'headline indicators.'
    ),
    'tags': 'wildfires, clustering, Lake Mead, metrics, dashboard',
    'type': 'CSV',
}
metrics_csv_path = processed_dir / 'mead_global_metrics.csv'
metrics_csv_item = gis.content.add(metrics_item_props, data=str(metrics_csv_path))
metrics_table_item = metrics_csv_item.publish()
print(f'Global metrics published: {metrics_table_item.title}')
print(f'  {metrics_table_item.homepage}')

c:\Users\mpp24\anaconda3\envs\obstacle-clustering\Lib\site-packages\IPython\core\interactiveshell.py:3701: DeprecatedWarning: add is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.add()` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Cluster summary published: Lake Mead Wildfires - Cluster Summary
  https://Michele-75.maps.arcgis.com/home/item.html?id=afefcc6ce483407bbf0c9e60d5c7d7af


c:\Users\mpp24\anaconda3\envs\obstacle-clustering\Lib\site-packages\IPython\core\interactiveshell.py:3701: DeprecatedWarning: add is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.add()` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Global metrics published: Lake Mead Wildfires - Global Metrics
  https://Michele-75.maps.arcgis.com/home/item.html?id=4a2643c7ab204308b5d6b712c6c83735


### 5.3 Share Publicly

Share all three items with everyone so the dashboard works for viewers without an ArcGIS account.

In [10]:
for item in [fire_layer_item, summary_table_item, metrics_table_item]:
    item.share(everyone=True)
    print(f'Shared: {item.title}')

print()
print('Item URLs (copy these for the dashboard build below):')
print(f'  Feature layer:   {fire_layer_item.homepage}')
print(f'  Cluster summary: {summary_table_item.homepage}')
print(f'  Global metrics:  {metrics_table_item.homepage}')

c:\Users\mpp24\anaconda3\envs\obstacle-clustering\Lib\site-packages\IPython\core\interactiveshell.py:3701: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Shared: Lake Mead Wildfires - Obstacle-Aware vs Standard k-Means


c:\Users\mpp24\anaconda3\envs\obstacle-clustering\Lib\site-packages\IPython\core\interactiveshell.py:3701: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Shared: Lake Mead Wildfires - Cluster Summary


c:\Users\mpp24\anaconda3\envs\obstacle-clustering\Lib\site-packages\IPython\core\interactiveshell.py:3701: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Shared: Lake Mead Wildfires - Global Metrics

Item URLs (copy these for the dashboard build below):
  Feature layer:   https://Michele-75.maps.arcgis.com/home/item.html?id=6c75d6ddde1548d3aba8875a0efafcc0
  Cluster summary: https://Michele-75.maps.arcgis.com/home/item.html?id=afefcc6ce483407bbf0c9e60d5c7d7af
  Global metrics:  https://Michele-75.maps.arcgis.com/home/item.html?id=4a2643c7ab204308b5d6b712c6c83735


In [14]:
#Lake Mead boundary as a feature layer

boundary_df = pd.read_csv(Path('../data/boundaries/mead_boundary.csv'))
coords = list(zip(boundary_df['longitude'], boundary_df['latitude']))

polygon = Polygon(coords)
gdf = gpd.GeoDataFrame(
    {'name': ['Lake Mead']},
    geometry=[polygon],
    crs='EPSG:4326'   # WGS 84
)

# Save to a shapefile (creates .shp + .dbf + .prj + .shx, all needed)
shp_dir = Path('../data/boundaries/mead_boundary_shp')
shp_dir.mkdir(exist_ok=True)
gdf.to_file(shp_dir / 'mead_boundary.shp')

# Zip the directory so AGOL can ingest it
import shutil
zip_path = Path('../data/boundaries/mead_boundary')
shutil.make_archive(str(zip_path), 'zip', shp_dir)
print(f'Saved zipped shapefile to {zip_path}.zip')

Saved zipped shapefile to ..\data\boundaries\mead_boundary.zip


## 6. Building the Standard $k$-Means Web Map

This is the first of two web maps. Both use the same feature layer (the fires we just published), but each styles the points by a different cluster column. The standard $k$-means map colors by `Cluster_Std`; the obstacle-aware map (next section) colors by `Cluster_OA`.

### Step 1: Open the layer in the Map Viewer

1. Go to the feature layer's item page (printed above as "Feature layer").
2. Click **Open in Map Viewer** in the top right.
3. The layer loads with default symbology.

### Step 2: Style the clusters by `Cluster_Std`

1. With the fires layer selected in the **Layers** panel, click **Styles** in the right rail.
2. Under **Choose attributes**, click **+ Field** and pick `Cluster_Std`. The Map Viewer auto-detects this as a category field.
3. Under **Pick a style**, choose **Types (unique symbols)** and click **Style options**.
4. Click each cluster's symbol to set its color from the project palette:
   - Cluster 1 -> `#e74c3c` (red)
   - Cluster 2 -> `#3498db` (blue)
   - Cluster 3 -> `#2ecc71` (green)
   - Cluster 4 -> `#f39c12` (orange)
5. For each cluster symbol, set:
   - **Size**: 8 px
   - **Outline color**: dark grey (`#282828` or similar)
   - **Outline width**: 0.5 px
   - **Transparency**: ~14%
6. Click **Done** on each panel until the Styles pane closes.

### Step 3: Configure the popup

1. With the fires layer selected, click **Pop-ups** in the right rail and toggle popups on.
2. Set the **Title** to `Fire in Cluster {Cluster_Std} (standard k-means)`.
3. Click **Fields list** and reorder/hide fields so the visible ones are:
   - Cluster_Std (label: "Standard Cluster")
   - Cluster_OA (label: "Obstacle-Aware Cluster")
   - Cause
   - Specific_Cause (label: "Specific Cause")
   - Fire_Size_Acres (label: "Fire Size (acres)", format to 2 decimal places)
   - Fire_Year (label: "Year")
4. Hide LATITUDE, LONGITUDE, and any auto-added ObjectID fields.

### Step 4: Set the basemap and extent

1. Click **Basemap** in the left rail. Pick **Topographic** -- the desert terrain context helps the comparison.
2. Pan/zoom so Lake Mead fills the view (rough center ~ 36.15 N, -114.4 W).
3. Once the extent looks right, click **Save and open** -> **Save as** in the top toolbar.

### Step 5: Save the web map

1. **Title**: `Lake Mead Wildfires - Standard k-Means Web Map`
2. **Tags**: `wildfires, clustering, Lake Mead, k-means, portfolio`
3. **Summary**: `Lake Mead wildfires (1992-2020) clustered with standard k-means on geographic coordinates only.`
4. Click **Save**.

### Step 6: Share publicly

1. From the web map's item page, click **Share** -> set sharing to **Everyone (public)**.
2. Copy the web map URL into the cell below.

In [ ]:
# Paste your standard k-means web map URL here once you've finished building it
std_webmap_url = ''
print(f'Standard k-Means web map URL: {std_webmap_url}')

## 7. Building the Obstacle-Aware Web Map

Same recipe as the standard map, with two differences: it colors by `Cluster_OA` instead of `Cluster_Std`, and the popup title reflects the method.

### Step 1-3: Same layer, different style field

1. Open the same feature layer in the Map Viewer (or use **Save as** from the standard k-means web map and edit).
2. **Styles** -> **+ Field** -> pick `Cluster_OA` (not `Cluster_Std`).
3. Apply the same Types (unique symbols) styling with the same four colors:
   - Cluster 1 -> `#e74c3c` (red)
   - Cluster 2 -> `#3498db` (blue)
   - Cluster 3 -> `#2ecc71` (green)
   - Cluster 4 -> `#f39c12` (orange)
4. Same symbol settings as before (size 8 px, outline `#282828`, etc.)

### Step 4: Configure the popup

Same fields visible as the standard map, but update the popup title:

1. **Title**: `Fire in Cluster {Cluster_OA} (obstacle-aware)`
2. Same field list (showing both Cluster_Std and Cluster_OA so viewers can see how the same fire is labeled by both methods).

### Step 5-6: Set basemap, extent, and save

1. Use the same basemap (Topographic) and the same extent as the standard k-means map -- consistency between the two maps is important for the side-by-side comparison.
2. Save as: `Lake Mead Wildfires - Obstacle-Aware Web Map`
3. **Tags**: `wildfires, clustering, Lake Mead, obstacle-aware, portfolio`
4. **Summary**: `Lake Mead wildfires (1992-2020) clustered with obstacle-aware k-means (β = 1.85).`
5. Click **Save**.

### Step 7: Share publicly

Same as before -- **Share** -> **Everyone (public)**.

In [ ]:
# Paste your obstacle-aware web map URL here once you've finished building it
oa_webmap_url = ''
print(f'Obstacle-Aware web map URL: {oa_webmap_url}')

## 8. Building the Comparison Dashboard

The dashboard is the interactive deliverable. With both web maps and the supporting tables in place, the layout work below takes about ten to fifteen minutes.

### Target layout

```
+------------------------------------------------------------+
|  Lake Mead Wildfires -- Standard vs Obstacle-Aware         |
|  An obstacle-aware k-means variant respects Lake Mead      |
|  as a barrier, producing tighter clusters along the lake.  |
+----------------+---------------------+---------------------+
|   Standard     |     Obstacle-Aware  |                     |
|   k-Means      |     k-Means         |  Span improvement:  |
|                |                     |     9.6%            |
|                |                     |                     |
|     [MAP]      |       [MAP]         |  Total fires: 844   |
|                |                     |                     |
|                |                     |                     |
| Cluster Legend | Cluster Legend      |  Cause: [pie chart] |
|                |                     |                     |
| Mean Fire Size | Mean Fire Size      |  Median Fire Size   |
+----------------+---------------------+---------------------+
| Source: FPA FOD 6th Edition (USFS) | Method: obstacle-aware|
+------------------------------------------------------------+
```

The two maps occupy most of the space. The middle/right column holds the headline indicator (span improvement), total fire count, and attribute breakdowns. The legend lists below each map filter that map's view when a cluster is clicked.

### Step 1: Create the dashboard

1. Open `https://michele-75.maps.arcgis.com/apps/dashboards/` and click **New dashboard**.
2. Title it **Lake Mead Wildfires - Standard vs Obstacle-Aware Clustering** and click **Create dashboard**.

### Step 2: Add the two maps

1. In the dashboard editor, click **+** (top left) -> **Map**.
2. Pick **Lake Mead Wildfires - Standard k-Means Web Map** for the left map.
3. On the configuration panel, enable **Pop-ups** and **Legend**. Leave other defaults.
4. Click **Done**.
5. Repeat for the right side: click **+** -> **Map** -> pick **Lake Mead Wildfires - Obstacle-Aware Web Map** -> Done.
6. Drag the second map to position it to the right of the first, leaving room on the right side for the indicators.

### Step 3: Add the headline indicator (span improvement)

This is the dashboard's main story: how much obstacle-aware clustering improves arc-length span over standard $k$-means.

1. Click **+** -> **Indicator**.
2. Data source: **Lake Mead Wildfires - Global Metrics**.
3. No filter needed (only one row in this table).
4. **Indicator** tab:
   - **Value**: `span_improvement_pct_basin`
   - **Statistic**: `Sum` (only one row, so sum equals the value)
   - **Decimal places**: 1
5. **General** tab:
   - **Top text**: `Obstacle-aware improvement`
   - **Middle text suffix**: `%`
   - **Bottom text**: `over standard k-means`
   - **Background color**: `#f59e0b` (amber) -- this makes the headline pop
   - **Text color**: `#1a1a1a` (near-black)
6. Click **Done**.
7. Position the indicator at the top of the right column.

### Step 4: Add the total fires indicator

1. Click **+** -> **Indicator**.
2. Data source: **Lake Mead Wildfires - Global Metrics**.
3. **Value**: `total_fires`. **Statistic**: `Sum`. **Decimal places**: 0.
4. **Top text**: `Total fires (1992-2020)`.
5. **Bottom text**: leave blank.
6. Click **Done**. Position below the headline indicator.

### Step 5: Add the cluster legend lists (one per map)

Each list shows the four clusters for its map and filters that map's data when a cluster is clicked.

**Left list (for the standard map)**:

1. Click **+** -> **List**.
2. Data source: **Lake Mead Wildfires - Cluster Summary**.
3. **Filter** tab: add a filter where `method is std`.
4. **List** tab -> **Line item template**, paste:
   ```
   <b>Cluster {cluster_label}</b><br>
   n = {n_fires} fires &nbsp;|&nbsp; {pct_human}% human-caused
   ```
   Set **Maximum results** to 10 (covers our 4 rows comfortably).
5. **General** tab: **Title** = `Standard k-means clusters`.
6. **Actions** tab: in the **Filter** section, toggle on the standard $k$-means map. Set the source field to `cluster_label` and the target field to `Cluster_Std`.
7. Click **Done**. Position below the standard map.

**Right list (for the obstacle-aware map)**:

1. Same as above but **Filter** = `method is oa_opt`.
2. **Title** = `Obstacle-aware clusters`.
3. **Actions** tab: target the obstacle-aware map instead. Source field `cluster_label`, target field `Cluster_OA`.
4. Position below the obstacle-aware map.

### Step 6: Add the per-cluster mean fire size indicators

These sit below the legends and show the mean fire size for the currently-selected cluster (or all fires when no cluster is selected).

**Left indicator (standard method)**:

1. Click **+** -> **Indicator**.
2. Data source: **Lake Mead Wildfires - Cluster Summary**.
3. **Filter** tab: `method is std`.
4. **Indicator** tab:
   - **Value**: `mean_fire_size`
   - **Statistic**: `Mean` (mean of one value = that value; mean across all four rows = the basin's average per-cluster mean)
   - **Decimal places**: 1
5. **General** tab:
   - **Top text**: `Mean fire size`
   - **Middle text suffix**: ` acres`
   - **Bottom text**: leave blank
6. **Actions** tab: enable this indicator as a filter target from the standard $k$-means legend list (Step 5). The same-data-source filter is automatic -- no field mapping needed.
7. Click **Done**.

**Right indicator (obstacle-aware method)**: same as above but `method is oa_opt`, and enable the filter target from the obstacle-aware legend list.

### Step 7: Add the fire cause pie chart

The pie chart sits in the right column under the total fires indicator. It shows the overall human vs. natural breakdown across all fires.

1. Click **+** -> **Pie chart**.
2. Data source: **Lake Mead Wildfires - Obstacle-Aware vs Standard k-Means** (the fires feature layer).
3. **Categories from**: pick **Grouped values** -> **Cause**.
4. **Sort by**: **Category** -> **Ascending** (locks order: Human-Caused first, then Natural -- avoids the chart flipping when filtered).
5. **General** tab:
   - **Title**: `Cause`
   - Custom colors:
     - Human-Caused: `#3a3a3a` (charcoal)
     - Natural: `#8b6f47` (warm brown / scorched-wood)
6. Click **Done**. Position in the right column.

### Step 8: Style the dashboard

A few visual touches that make a big difference:

1. **Page header** (the top bar):
   - **Background color**: `#2c3e50` (slate)
   - **Text color**: `#f4f1ec` (warm off-white)
   - **Title**: `Lake Mead Wildfires - Standard vs Obstacle-Aware Clustering`
   - **Subtitle**: `An obstacle-aware k-means variant respects Lake Mead as a barrier; the boundary parameter` *s* `improves arc-length span by 9.6% on the full basin.`
2. **Page background**: `#f4f1ec`
3. **Indicator element backgrounds**: `#faf7f1` (warm off-white) for all non-headline indicators
4. **List element backgrounds**: same `#faf7f1`
5. **Body text color** throughout: `#2c3e50`

### Step 9: Add the footer

1. Click **+** -> **Rich text**. Position at the bottom of the dashboard (or use the bottom panel if your Dashboards version exposes it).
2. Set the content to:
   ```
   Wildfire data: FPA FOD 6th Edition (USFS), 1992-2020, fires with known causes only
   | Method: obstacle-aware k-means | Author: Michele Perry | [GitHub repository link]
   ```
3. Left-align. Small text (11-12 px). Muted color (`#6b7280`).

### Step 10: Save and share

1. Click **Save** in the dashboard toolbar.
2. From the dashboard's item page, set sharing to **Everyone (public)**.
3. Copy the dashboard URL into the cell below.

In [ ]:
# Paste your dashboard URL here once you've finished building it
dashboard_url = ''
print(f'Dashboard URL: {dashboard_url}')

## 9. Updating Items In Place (Optional)

If you re-run Notebook 03 and produce new CSVs, you can update the existing AGOL items without breaking any dashboard configurations that reference them. This is the "Update Data" flow from the AGOL UI but run programmatically.

The code below finds the existing items by title and replaces their data with the latest local CSVs. Item IDs and any sharing/configuration metadata are preserved.

In [ ]:
def update_existing_item(title, local_csv_path):
    """Find an existing item by title and replace its data."""
    me = gis.users.me.username
    matches = gis.content.search(
        query=f'title:"{title}" owner:{me}',
        max_items=20,
    )
    matches = [it for it in matches if it.title == title]

    if not matches:
        print(f'  No item found with title "{title}". Skipping update.')
        return None

    if len(matches) > 1:
        print(f'  Multiple items found with title "{title}". '
              f'Updating the first one ({matches[0].id}).')

    item = matches[0]
    print(f'  Updating item: {item.title} (ID {item.id})')
    item.update(data=str(local_csv_path))
    return item


# Uncomment the block below to perform the update
# print('Updating fires feature layer source CSV...')
# update_existing_item(
#     'Lake Mead Wildfires - Obstacle-Aware vs Standard k-Means',
#     publish_path,
# )

# print('\nUpdating cluster summary table...')
# update_existing_item(
#     'Lake Mead Wildfires - Cluster Summary',
#     processed_dir / 'mead_cluster_summary.csv',
# )

# print('\nUpdating global metrics table...')
# update_existing_item(
#     'Lake Mead Wildfires - Global Metrics',
#     processed_dir / 'mead_global_metrics.csv',
# )

## 10. Summary

What this notebook accomplished:

1. Loaded the three CSVs produced by Notebook 03
2. Prepared a publishing-ready version of the fires CSV with friendly column names and labels
3. Uploaded all three CSVs to ArcGIS Online and published them as hosted items (feature layer + two tables)
4. Provided detailed step-by-step instructions for building the two web maps (standard k-means and obstacle-aware) and the comparison dashboard
5. Included an optional Update Data flow for re-running Notebook 03 and refreshing the AGOL items in place

The dashboard's story: same 844 Lake Mead wildfires, two clustering methods, visible difference in how clusters relate to the lake. Standard $k$-means produces clusters that don't respect the water; obstacle-aware $k$-means produces clusters that do, and tightens arc-length span by ~9.6% in the process.

Lake Tahoe's results (Notebook 02) provide the contrast for the writeup, but the dashboard focuses on Lake Mead alone for visual clarity.